In [ ]:
# SETUP

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install tools
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'samtools', 'bedtools'],
               capture_output=True)
subprocess.run(['pip', 'install', 'pysam', 'pybedtools', 'scikit-learn'],
               capture_output=True)

# ChromHMM path
CHROMHMM = '/content/drive/MyDrive/chromatin_project/ChromHMM/ChromHMM.jar'

import os
print(f"ChromHMM: {'✓' if os.path.exists(CHROMHMM) else '✗ not found'}")
print("Setup complete")

In [ ]:
!java -mx4000M \
    -jar /content/drive/MyDrive/chromatin_project/ChromHMM/ChromHMM.jar \
    LearnModel \
    -p 0 \
    -init random \
    -s 42 \
    -nobrowser \
    -r 500 \
    /content/drive/MyDrive/chromatin_project/binarized/ \
    /content/drive/MyDrive/chromatin_project/chromhmm_output/ \
    10 \
    hg38

In [ ]:

import pysam
import numpy as np

DRIVE = '/content/drive/MyDrive/chromatin_project'
MARKS = ['H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me3', 'H3K9me3']
BIN_SIZE = 200
chrom = 'chr18'

bam_dir = f'{DRIVE}/bam_files'
cols = []
for mark in MARKS:
    bam = pysam.AlignmentFile(f'{bam_dir}/{mark}.bam', 'rb')
    chrom_len = dict(zip(bam.references, bam.lengths))[chrom]
    n_bins = chrom_len // BIN_SIZE
    counts = np.zeros(n_bins, dtype=np.int32)
    for read in bam.fetch(chrom):
        if read.is_unmapped or read.is_duplicate:
            continue
        idx = read.reference_start // BIN_SIZE
        if idx < n_bins:
            counts[idx] += 1
    bam.close()
    print(f"{mark}: {n_bins} bins, mean={counts.mean():.2f}, max={counts.max()}")
    cols.append(counts)

arr = np.stack(cols, axis=1)
out = f'{DRIVE}/count_matrix_chr18.npy'
np.save(out, arr)
print(f"\nSaved to {out}")
